# 05 — LSTM Autoencoder

## 1. Objective

Develop an LSTM autoencoder for unsupervised anomaly detection on the SWaT multivariate time series.

The model will be trained using normal calibration data and will learn to reconstruct temporal patterns in the sensor measurements. Reconstruction error will then provide an anomaly score: observations that are poorly reconstructed relative to the learned normal behaviour may indicate anomalous behaviour.

The implementation follows the LSTM autoencoder approach used in the primary reference notebook, while maintaining the project's requirements for chronological data handling, separation of calibration and evaluation data, reproducibility, and prevention of data leakage.

Attack labels will not be used during model training or reconstruction-error threshold calibration.

### Scope of Evaluation

This notebook evaluates the LSTM autoencoder on the normal validation period using the same eligible-normal split as Notebook 04. Attack-side evaluation — including confusion-matrix counts, precision, recall, and F1 on attack-labelled sequences — is performed in a subsequent stage, following the deferral recorded in Notebook 04 §13.3. The temporal overlap between the attack file and the normal time windows, established by Notebook 02's normal–attack timestamp overlap analysis, must be respected when that evaluation is carried out.

## 2. Methodology

An autoencoder is trained to reconstruct input observations from a compressed internal representation. When trained only on normal process behaviour, the model is expected to learn the temporal patterns present in normal operation.

For an input sequence X, the LSTM encoder maps the sequence to a lower-dimensional representation:

X → Encoder → Z

The decoder then reconstructs the sequence:

Z → Decoder → X̂

The reconstruction error measures the difference between the original and reconstructed sequence. For a sequence containing T time steps and D sensor features, the mean squared reconstruction error is:

MSE = (1 / (T × D)) × Σ (Xₜ,ⱼ − X̂ₜ,ⱼ)²

where:

- Xₜ,ⱼ is the observed value for feature j at time step t.
- X̂ₜ,ⱼ is the corresponding reconstructed value.
- T is the sequence length.
- D is the number of sensor features.

A threshold derived from the reconstruction-error distribution on held-out normal calibration observations is then used to classify sequences as normal or anomalous. Following the percentile approach used by the simple statistical baseline in Notebook 04 §12, the threshold is set at the 99th percentile of reconstruction errors on a portion of the calibration period held out from model training. The threshold is fixed before any attack sequence is scored, and attack labels are not used to select or adjust it. The specific hold-out split used for threshold calibration is recorded alongside the model-training code.

### Temporal Windowing

The model operates on sequences rather than isolated observations. Each sequence contains a fixed number of consecutive observations while preserving their chronological order.

The primary reference implementation uses a window size of 30 observations. This project will initially retain that value so that the implementation remains comparable to the reference, unless the data or modelling results provide evidence that a change is necessary.

Training sequences are generated from normal calibration data only. Normal-validation sequences are chronologically disjoint from the training sequences. Attack sequences are evaluated separately and are known to overlap the calibration and validation time windows, even though they share no exact timestamps with them. Attack-side evaluation is therefore treated as a distinct, time-window-overlapping evaluation, not as a temporally independent hold-out.

The training, normal-validation and threshold-calibration data use the eligible-normal split established in Notebook 04 §9 as their outer boundary: 316,238 calibration observations (2015-12-28 10:00:00 → 2016-01-01 15:51:19) and 79,060 normal-validation observations (2016-01-01 15:51:20 → 2016-01-02 14:59:59), drawn from the 395,298 eligible normal observations. Within this boundary, the calibration period is further divided chronologically as described in Section 3.2, so that threshold calibration uses data held out from model training. No observation from the normal-validation period contributes to training, model selection, or threshold calibration.

Sequence generation respects the calibration/validation boundary: the last training sequence ends strictly before the first validation observation, so no validation observation contributes to the reconstruction-error statistics on the calibration data. Within the calibration period, sequences are generated with the configured stride; at the boundary, sequences are truncated to avoid crossing into the validation period.

### Input Feature Contract

Notebook 04 §12.5 documented that nine sensors — `P202`, `P204`, `P206`, `P301`, `P401`, `P404`, `P502`, `P601`, `P603` — had zero standard deviation during the calibration period, and Notebook 03 §6.2 identified six of these as globally constant across both the normal and attack datasets. Zero-variance features produce no reconstruction-error signal and contribute no gradient to the autoencoder.

This notebook excludes the nine constant-in-calibration sensors from the LSTM input, giving a 42-feature input contract. This choice mirrors the 42-feature evaluation set used by the simple statistical threshold baseline in Notebook 04 §12.5 and eliminates zero-gradient inputs from the training signal. The 42 retained features are the remaining sensor columns after the nine excluded features are removed. The input contract is recorded here so that the anomaly scores reported subsequently are reproducible.

### Distribution-Shift Risk

Notebook 04 §10.4 found that the rule-based detector, calibrated on the first 80% of the eligible normal data, flagged 92.71% of the remaining 20% of normal observations. This indicates a substantial shift in the marginal distributions of several sensors between the calibration and validation periods. An LSTM autoencoder trained on calibration data may produce systematically higher reconstruction errors on the validation period, in which case a threshold calibrated directly on the training reconstruction errors would over-flag normal validation observations.

To mitigate this risk, the anomaly threshold is calibrated on a portion of the calibration period held out from model training, so that the threshold reflects the model's reconstruction behaviour on data not seen during training. The distribution of reconstruction errors on the normal validation period is reported alongside the threshold so that the extent of any residual shift remains visible.

### Data Leakage Prevention

The preprocessing parameters used by the model must be learned from the normal calibration data only. Validation and attack observations must not influence the fitted scaler or the model parameters during training.

The final anomaly threshold is also established without using attack labels for threshold selection. All threshold values are derived from held-out normal-calibration reconstruction errors.

## 3. Reproducibility and Model Configuration

The LSTM autoencoder will be implemented in PyTorch. The initial configuration follows the primary reference notebook to maintain comparability with the reference implementation.

### 3.1 Model Configuration

| Parameter | Value |
|:--|--:|
| Random seed | 42 |
| Sequence window | 30 observations |
| Training stride | 5 observations |
| Evaluation stride | 1 observation |
| Hidden size | 64 |
| LSTM layers | 2 |
| Bottleneck size | 16 |
| Dropout | 0.2 |
| Loss function | Mean Squared Error |
| Optimizer | Adam |
| Initial learning rate | 0.001 |
| Batch size | 256 |
| Maximum training epochs | 50 |
| Learning-rate scheduler | ReduceLROnPlateau |
| Learning-rate reduction factor | 0.5 |
| Scheduler patience | 3 epochs |
| Early-stopping patience | 5 epochs |

These settings define the initial experiment. They are not assumed to be universally optimal, and any justified changes will be recorded.

### 3.2 Calibration Data Separation

The normal calibration period established in Notebook 04 contains 316,238 observations. This period will be divided chronologically into three non-overlapping portions:

1. **Model-training data:** Used to update the model parameters.
2. **Training-monitoring data:** Used to monitor reconstruction loss, support learning-rate scheduling, and determine early stopping.
3. **Threshold-calibration data:** Kept separate from model fitting and training monitoring. Its reconstruction errors will determine the fixed 99th-percentile anomaly threshold.

The exact split proportions and observation counts will be recorded in the implementation.

The normal validation period established in Notebook 04 will remain separate from these calibration subsets. It will be used to assess reconstruction errors and detection behaviour after the model and threshold have been fixed.

Attack-labelled observations will not be used for model training, training monitoring, or threshold selection.

### 3.3 Training Procedure and Monitoring

Training will report, for each epoch, the epoch number, training loss, monitoring loss, and current learning rate. The training history — including per-epoch losses and learning rates — will be retained for subsequent analysis and visualisation.

The best model state, as determined by the lowest monitoring loss, will be preserved. Early stopping will halt training when the monitoring loss has not improved for the configured number of patience epochs.

Random seeds are set once in the environment-setup code cell before any model initialisation. Device selection, model configuration, and data-split boundaries are recorded in Sections 2 and 3.1, and are reflected in the training-time printouts so that a completed training run can be reconstructed from its log alone.

Reproducibility verification across devices and software environments is deferred to Notebook 12.

In [1]:
# 4. Environment Setup and Reproducibility

import sys
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# --------------------------------------------------
# 1. Reproducibility
# --------------------------------------------------

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Use deterministic cuDNN behaviour where available.
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# --------------------------------------------------
# 2. Locate the project root
# --------------------------------------------------

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    (
        directory
        for directory in (CURRENT_DIR, *CURRENT_DIR.parents)
        if (directory / "data" / "raw" / "normal.csv").is_file()
        and (directory / "data" / "raw" / "attack.csv").is_file()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'data/raw/normal.csv' and 'data/raw/attack.csv'. "
        "Open Jupyter from within the project directory "
        "or one of its subdirectories."
    )

DATA_ROOT = PROJECT_ROOT / "data" / "raw"

# --------------------------------------------------
# 3. Select the available PyTorch device
# --------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --------------------------------------------------
# 4. Report the environment
# --------------------------------------------------

print("ENVIRONMENT SETUP")
print("-" * 50)
# print(f"Project root:       {PROJECT_ROOT}")
# print(f"Raw data directory: {DATA_ROOT}")
print(f"Normal file found:  {(DATA_ROOT / 'normal.csv').is_file()}")
print(f"Attack file found:  {(DATA_ROOT / 'attack.csv').is_file()}")
print()
print(f"Random seed:        {SEED}")
print(f"Python version:     {sys.version.split()[0]}")
print(f"PyTorch version:    {torch.__version__}")
print(f"NumPy version:      {np.__version__}")
print(f"Pandas version:     {pd.__version__}")
print(f"Selected device:    {DEVICE}")

if DEVICE.type == "cuda":
    print(f"GPU:                {torch.cuda.get_device_name(0)}")
else:
    print("GPU:                No CUDA-compatible GPU available to PyTorch.")
    print("                    Training will use the CPU.")

print("-" * 50)

ENVIRONMENT SETUP
--------------------------------------------------
Normal file found:  True
Attack file found:  True

Random seed:        42
Python version:     3.11.14
PyTorch version:    2.6.0+cpu
NumPy version:      2.4.6
Pandas version:     3.0.6
Selected device:    cpu
GPU:                No CUDA-compatible GPU available to PyTorch.
                    Training will use the CPU.
--------------------------------------------------


In [6]:
import warnings

# Suppress the specific date-parsing warning by message text
warnings.filterwarnings("ignore", message=".*Parsing dates.*")

# 1. Load the normal dataset

normal_path = DATA_ROOT / "normal.csv"

normal = pd.read_csv(normal_path, low_memory=False)

# Standardise column names without modifying the raw file.
normal.columns = normal.columns.str.strip()

# Identify the timestamp and label columns.
TIMESTAMP_COL = "Timestamp"

LABEL_COL = next(
    (
        column
        for column in normal.columns
        if column.lower() in {"normal/attack", "label"}
    ),
    None,
)

if TIMESTAMP_COL not in normal.columns:
    raise KeyError("The expected 'Timestamp' column was not found.")

if LABEL_COL is None:
    raise KeyError("The dataset label column was not found.")

# Identify the sensor features.
feature_cols = [
    column
    for column in normal.columns
    if column not in {TIMESTAMP_COL, LABEL_COL}
]

# Parse timestamps and identify complete observations.
normal[TIMESTAMP_COL] = pd.to_datetime(
    normal[TIMESTAMP_COL],
    errors="coerce",
)

complete_sensor_mask = normal[feature_cols].notna().all(axis=1)
valid_timestamp_mask = normal[TIMESTAMP_COL].notna()

eligible_mask = complete_sensor_mask & valid_timestamp_mask

eligible_normal = normal.loc[eligible_mask].copy()

# 2. Reconstruct the exact chronological split from Notebook 04

CALIBRATION_FRACTION = 0.80

n_eligible = len(eligible_normal)
calibration_end = int(n_eligible * CALIBRATION_FRACTION)

normal_calibration = eligible_normal.iloc[:calibration_end].copy()
normal_validation = eligible_normal.iloc[calibration_end:].copy()

# 3. Verify the reconstructed data

print("NORMAL DATA PREPARATION")
print("-" * 60)

print(f"Normal dataset shape:          {normal.shape}")
print(f"Number of sensor features:     {len(feature_cols)}")
print(f"Valid timestamps:              {valid_timestamp_mask.sum():,}")
print(f"Complete sensor observations:  {complete_sensor_mask.sum():,}")
print(f"Eligible observations:         {n_eligible:,}")

print("\nCALIBRATION AND VALIDATION SPLIT")
print("-" * 60)

print(f"Calibration observations:      {len(normal_calibration):,}")
print(f"Validation observations:       {len(normal_validation):,}")

print(
    "Calibration period:           "
    f"{normal_calibration[TIMESTAMP_COL].iloc[0]} "
    "to "
    f"{normal_calibration[TIMESTAMP_COL].iloc[-1]}"
)

print(
    "Validation period:            "
    f"{normal_validation[TIMESTAMP_COL].iloc[0]} "
    "to "
    f"{normal_validation[TIMESTAMP_COL].iloc[-1]}"
)

print("\nVERIFICATION")
print("-" * 60)

print(
    "Calibration precedes validation:",
    normal_calibration[TIMESTAMP_COL].iloc[-1]
    < normal_validation[TIMESTAMP_COL].iloc[0],
)

print(
    "Calibration contains only normal labels:",
    normal_calibration[LABEL_COL].eq("Normal").all(),
)

print(
    "Validation contains only normal labels:",
    normal_validation[LABEL_COL].eq("Normal").all(),
)

print("-" * 60)

NORMAL DATA PREPARATION
------------------------------------------------------------
Normal dataset shape:          (1387098, 53)
Number of sensor features:     51
Valid timestamps:              1,387,093
Complete sensor observations:  395,298
Eligible observations:         395,298

CALIBRATION AND VALIDATION SPLIT
------------------------------------------------------------
Calibration observations:      316,238
Validation observations:       79,060
Calibration period:           2015-12-28 10:00:00 to 2016-01-01 15:51:19
Validation period:            2016-01-01 15:51:20 to 2016-01-02 14:59:59

VERIFICATION
------------------------------------------------------------
Calibration precedes validation: True
Calibration contains only normal labels: True
Validation contains only normal labels: True
------------------------------------------------------------


In [7]:
# 6. Define the LSTM Input Feature Contract

EXCLUDED_FEATURES = [
    "P202",
    "P204",
    "P206",
    "P301",
    "P401",
    "P404",
    "P502",
    "P601",
    "P603",
]


# Verify that every excluded feature exists.
missing_excluded = [
    feature for feature in EXCLUDED_FEATURES
    if feature not in feature_cols
]

if missing_excluded:
    raise KeyError(
        "The following expected zero-variance calibration features "
        f"were not found: {missing_excluded}"
    )

# Retain the remaining sensor features.
model_feature_cols = [
    feature for feature in feature_cols
    if feature not in EXCLUDED_FEATURES
]

# Verify the expected feature count.
if len (model_feature_cols) != 42:
    raise ValueError(
        f"Expected 42 model features, found {len(model_feature_cols)}."
    )


# Confirm that the excluded features have zero variance
# in the normal calibration period.

calibration_std = normal_calibration[EXCLUDED_FEATURES].std(ddof=1)

nonzero_excluded_std = calibration_std[calibration_std > 0]

if not nonzero_excluded_std.empty:
    raise ValueError(
        "One or more excluded features are not constant in the "
        "normal calibration period:\n"
        f"{nonzero_excluded_std}"
    )

print("LSTM INPUT FEATURE CONTRACT")
print("-" * 60)
print(f"Original sensor features:     {len(feature_cols)}")
print(f"Excluded features:            {len(EXCLUDED_FEATURES)}")
print(f"Model input features:         {len(model_feature_cols)}")
print()
print("Excluded features:")
print(", ".join(EXCLUDED_FEATURES))
print()
print("Model input contract verified:", len(model_feature_cols) == 42)

LSTM INPUT FEATURE CONTRACT
------------------------------------------------------------
Original sensor features:     51
Excluded features:            9
Model input features:         42

Excluded features:
P202, P204, P206, P301, P401, P404, P502, P601, P603

Model input contract verified: True


In [8]:
# 7. Internal Calibration Split

TRAIN_FRACTION = 0.70
MONITOR_FRACTION = 0.15
THRESHOLD_FRACTION = 0.15

n_calibration = len(normal_calibration)

train_end = int(n_calibration * TRAIN_FRACTION)
monitor_end = train_end + int(n_calibration * MONITOR_FRACTION)

lstm_train = normal_calibration.iloc[:train_end].copy()
lstm_monitor = normal_calibration.iloc[train_end:monitor_end].copy()
lstm_threshold = normal_calibration.iloc[monitor_end:].copy()

print("LSTM INTERNAL CALIBRATION SPLIT")
print("-" * 60)

print(f"Total calibration observations: {n_calibration:,}")
print()

print(
    f"Training observations:          {len(lstm_train):,} "
    f"({len(lstm_train) / n_calibration:.2%})"
)
print(
    f"Monitoring observations:        {len(lstm_monitor):,} "
    f"({len(lstm_monitor) / n_calibration:.2%})"
)
print(
    f"Threshold-calibration obs.:     {len(lstm_threshold):,} "
    f"({len(lstm_threshold) / n_calibration:.2%})"
)

print()
print("TIME PERIODS")
print("-" * 60)

print(
    "Training:          "
    f"{lstm_train[TIMESTAMP_COL].iloc[0]} "
    f"to {lstm_train[TIMESTAMP_COL].iloc[-1]}"
)

print(
    "Monitoring:        "
    f"{lstm_monitor[TIMESTAMP_COL].iloc[0]} "
    f"to {lstm_monitor[TIMESTAMP_COL].iloc[-1]}"
)

print(
    "Threshold:         "
    f"{lstm_threshold[TIMESTAMP_COL].iloc[0]} "
    f"to {lstm_threshold[TIMESTAMP_COL].iloc[-1]}"
)

print()
print("BOUNDARY VERIFICATION")
print("-" * 60)

print(
    "Training precedes monitoring:",
    lstm_train[TIMESTAMP_COL].iloc[-1]
    < lstm_monitor[TIMESTAMP_COL].iloc[0]
)

print(
    "Monitoring precedes threshold calibration:",
    lstm_monitor[TIMESTAMP_COL].iloc[-1]
    < lstm_threshold[TIMESTAMP_COL].iloc[0]
)

print(
    "All three sets contain only normal labels:",
    (
        lstm_train[LABEL_COL].eq("Normal").all()
        and lstm_monitor[LABEL_COL].eq("Normal").all()
        and lstm_threshold[LABEL_COL].eq("Normal").all()
    )
)

LSTM INTERNAL CALIBRATION SPLIT
------------------------------------------------------------
Total calibration observations: 316,238

Training observations:          221,366 (70.00%)
Monitoring observations:        47,435 (15.00%)
Threshold-calibration obs.:     47,437 (15.00%)

TIME PERIODS
------------------------------------------------------------
Training:          2015-12-28 10:00:00 to 2015-12-31 12:43:42
Monitoring:        2015-12-31 12:43:43 to 2016-01-01 02:23:17
Threshold:         2016-01-01 02:23:18 to 2016-01-01 15:51:19

BOUNDARY VERIFICATION
------------------------------------------------------------
Training precedes monitoring: True
Monitoring precedes threshold calibration: True
All three sets contain only normal labels: True


In [9]:
# 8. Fit the Scaler on Training Data Only

from sklearn.preprocessing import MinMaxScaler

# Extract the 42 model features from each chronological subset.
X_train_raw = lstm_train[model_feature_cols].astype(np.float32)
X_monitor_raw = lstm_monitor[model_feature_cols].astype(np.float32)
X_threshold_raw = lstm_threshold[model_feature_cols].astype(np.float32)
X_validation_raw = normal_validation[model_feature_cols].astype(np.float32)

# Confirm that the selected inputs contain no missing or infinite values.

datasets_to_check = {
    "Training": X_train_raw,
    "Monitoring": X_monitor_raw,
    "Threshold calibration": X_threshold_raw,
    "Normal validation": X_validation_raw,
}

for dataset_name, dataset in datasets_to_check.items():
    values = dataset.to_numpy()

    if not np.isfinite(values).all():
        raise ValueError(
            f"{dataset_name} features contain missing or infinite values."
        )

# Fit only on the model-training subset.
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_raw)

# Apply the fitted scaler without refitting.
X_monitor = scaler.transform(X_monitor_raw)
X_threshold = scaler.transform(X_threshold_raw)
X_validation = scaler.transform(X_validation_raw)

# Verify shapes and scaling behaviour.
print("SCALING VERIFICATION")
print("-" * 60)

print(f"Training array shape:            {X_train.shape}")
print(f"Monitoring array shape:          {X_monitor.shape}")
print(f"Threshold-calibration shape:     {X_threshold.shape}")
print(f"Normal-validation shape:         {X_validation.shape}")

print()
print(
    "Training values within [0, 1]:",
    bool((X_train >= 0).all() and (X_train <= 1).all())
)

print(
    "Monitoring values outside [0, 1]:",
    int(((X_monitor < 0) | (X_monitor > 1)).sum())
)

print(
    "Threshold values outside [0, 1]:",
    int(((X_threshold < 0) | (X_threshold > 1)).sum())
)

print(
    "Normal-validation values outside [0, 1]:",
    int(((X_validation < 0) | (X_validation > 1)).sum())
)

print()
print("Scaler fitted on training data only: True")
print("Feature count consistent across subsets:", 
      X_train.shape[1] == X_monitor.shape[1]
      == X_threshold.shape[1] == X_validation.shape[1])

SCALING VERIFICATION
------------------------------------------------------------
Training array shape:            (221366, 42)
Monitoring array shape:          (47435, 42)
Threshold-calibration shape:     (47437, 42)
Normal-validation shape:         (79060, 42)

Training values within [0, 1]: True
Monitoring values outside [0, 1]: 71582
Threshold values outside [0, 1]: 114654
Normal-validation values outside [0, 1]: 261512

Scaler fitted on training data only: True
Feature count consistent across subsets: True


In [10]:
# 9. Define Sequence-Generation Parameters

WINDOW_SIZE = 30
TRAIN_STRIDE = 5
EVAL_STRIDE =1

# Basic parameter checks.
if WINDOW_SIZE <= 0:
    raise ValueError("WINDOW_SIZE must be positive.")

if TRAIN_STRIDE <= 0 or EVAL_STRIDE <= 0:
    raise ValueError("Sequence strides must be positive.")

if X_train.shape[1] != len(model_feature_cols):
    raise ValueError("Training feature count does not match the input contract.")

if any(
    len(array) < WINDOW_SIZE
    for array in (X_train, X_monitor, X_threshold, X_validation)
):
    raise ValueError("At least one subset is shorter than WINDOW_SIZE.")

print("SEQUENCE-GENERATION CONFIGURATION")
print("-" * 60)
print(f"Window size:              {WINDOW_SIZE}")
print(f"Training stride:           {TRAIN_STRIDE}")
print(f"Evaluation stride:         {EVAL_STRIDE}")
print(f"Features per observation:  {len(model_feature_cols)}")
print(f"Expected sequence shape:   (number of sequences, {WINDOW_SIZE}, {len(model_feature_cols)})")
print()
print("All subsets are long enough for sequence generation.")
print("Sequence parameters verified.")

SEQUENCE-GENERATION CONFIGURATION
------------------------------------------------------------
Window size:              30
Training stride:           5
Evaluation stride:         1
Features per observation:  42
Expected sequence shape:   (number of sequences, 30, 42)

All subsets are long enough for sequence generation.
Sequence parameters verified.


In [12]:
# 10. Sequence-Generation Function

def create_sequences(data, window_size, stride):
    """
    Create chronological sliding-window sequences.

    Parameters
    ----------
    data : np.ndarray
        Scaled observations with shape (n_observations, n_features).
    window_size : int
        Number of observations in each sequence.
    stride : int
        Step between consecutive sequence starting positions.

    Returns
    -------
    sequences : np.ndarray
        Array with shape (n_sequences, window_size, n_features).
    end_indices : np.ndarray
        Original subset indices corresponding to each sequence's
        final observation.
    """
    data = np.asarray(data)

    if data.ndim != 2:
        raise ValueError(
            f"Expected 2D input, received array with shape {data.shape}."
        )

    if window_size <= 0:
        raise ValueError("window_size must be positive.")

    if stride <= 0:
        raise ValueError("stride must be positive.")

    n_observations, n_features = data.shape

    if n_observations < window_size:
        raise ValueError(
            f"Input contains {n_observations} observations, "
            f"but window_size is {window_size}."
        )

    start_indices = np.arange(
        0,
        n_observations - window_size + 1,
        stride,
    )

    sequences = np.stack(
        [
            data[start:start + window_size]
            for start in start_indices
        ],
        axis=0,
    )

    end_indices = start_indices + window_size - 1

    return sequences.astype(np.float32), end_indices